In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except:
    XGB_AVAILABLE = False


class ForecastTrainer:

    def __init__(self, filepath):

        self.filepath = filepath

        os.makedirs("../outputs/forecasting/models", exist_ok=True)
        os.makedirs("../outputs/forecasting/metrics", exist_ok=True)

    ############################################################

    def load(self):

        print("=" * 60)
        print("Loading Forecast Dataset")
        print("=" * 60)

        self.df = pd.read_csv(
            self.filepath,
            parse_dates=["Date"]
        )

        print(self.df.shape)

    ############################################################

    def prepare_data(self):

        feature_columns = [

            "Quantity",
            "Orders",
            "Customers",
            "Avg_Order_Value",

            "Year",
            "Quarter",
            "Month",
            "Week",
            "Day",
            "DayOfWeek",
            "Weekend",

            "Lag_1",
            "Lag_7",
            "Lag_14",
            "Lag_30",

            "Rolling_Mean_7",
            "Rolling_Mean_14",
            "Rolling_Mean_30",

            "Rolling_STD_7",
            "Rolling_STD_30",

            "Revenue_Growth"

        ]

        self.X = self.df[feature_columns]

        self.y = self.df["Revenue"]

        split = int(len(self.df) * 0.80)

        self.X_train = self.X.iloc[:split]
        self.X_test = self.X.iloc[split:]

        self.y_train = self.y.iloc[:split]
        self.y_test = self.y.iloc[split:]

        print()

        print("Training Samples :", len(self.X_train))
        print("Testing Samples  :", len(self.X_test))

    ############################################################

    def evaluate(self, model, name):

        prediction = model.predict(self.X_test)

        mae = mean_absolute_error(
            self.y_test,
            prediction
        )

        rmse = np.sqrt(
            mean_squared_error(
                self.y_test,
                prediction
            )
        )

        mape = mean_absolute_percentage_error(
            self.y_test,
            prediction
        )

        print()

        print("=" * 40)
        print(name)
        print("=" * 40)

        print("MAE :", round(mae,2))
        print("RMSE:", round(rmse,2))
        print("MAPE:", round(mape*100,2),"%")

        return {

            "Model":name,
            "MAE":mae,
            "RMSE":rmse,
            "MAPE":mape

        }

    ############################################################

    def train_models(self):

        results=[]

        print("\nTraining Linear Regression...\n")

        lr=LinearRegression()

        lr.fit(
            self.X_train,
            self.y_train
        )

        results.append(
            self.evaluate(lr,"Linear Regression")
        )

        joblib.dump(
            lr,
            "../outputs/forecasting/models/linear.pkl"
        )

        #######################################################

        print("\nTraining Random Forest...\n")

        rf=RandomForestRegressor(

            n_estimators=300,

            random_state=42

        )

        rf.fit(

            self.X_train,

            self.y_train

        )

        results.append(

            self.evaluate(

                rf,

                "Random Forest"

            )

        )

        joblib.dump(

            rf,

            "../outputs/forecasting/models/random_forest.pkl"

        )

        #######################################################

        if XGB_AVAILABLE:

            print("\nTraining XGBoost...\n")

            xgb=XGBRegressor(

                n_estimators=300,

                learning_rate=0.05,

                max_depth=5,

                random_state=42

            )

            xgb.fit(

                self.X_train,

                self.y_train

            )

            results.append(

                self.evaluate(

                    xgb,

                    "XGBoost"

                )

            )

            joblib.dump(

                xgb,

                "../outputs/forecasting/models/xgboost.pkl"

            )

        #######################################################

        results=pd.DataFrame(results)

        results.to_csv(

            "../outputs/forecasting/metrics/model_metrics.csv",

            index=False

        )

        print("\nMetrics Saved")

        print("\nBest Model")

        print(

            results.sort_values(

                "MAPE"

            ).head(1)

        )

    ############################################################

    def run(self):

        self.load()

        self.prepare_data()

        self.train_models()


if __name__=="__main__":

    obj=ForecastTrainer(

        r"C:\Users\ka843\Coding\Amdox Internship_project\src\outputs\forecasting\datasets\forecast_features.csv"

    )

    obj.run()

Loading Forecast Dataset


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\ka843\\Coding\\Amdox Internship_project\\outputs\\forecasting\\datasets\\forecast_features.csv'